# Colab Dense Embedding & Indexing - LegalRAG

Notebook này tự động hóa 100% quá trình **tạo Dense Vector Index (Embedding)** cho dữ liệu Bộ luật Lao động 2019 sử dụng GPU (Tesla T4) trên Google Colab.

⚡ **Chỉ cần bấm: Runtime -> Run all (hoặc `Ctrl + F9`)** để chạy toàn bộ từ đầu đến cuối.

In [ ]:
# 1. Kiểm tra GPU và môi trường
!nvidia-smi
import torch

print("\n" + "=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU để tăng tốc.")
print("=" * 50)

In [ ]:
# 2. Clone mã nguồn LegalRAG và nạp thư viện
import os
import sys
from pathlib import Path

# Clone repo nếu chưa có, hoặc pull bản mới nhất
if not Path("/content/LegalRAG").exists():
    !git clone https://github.com/ngminhduc129/LegalRAG.git /content/LegalRAG
else:
    !cd /content/LegalRAG && git pull

%cd /content/LegalRAG

# Cài đặt các gói cần thiết
!pip install sentence-transformers underthesea rank-bm25 typer rich loguru --quiet

# Thêm src vào sys.path để import legal_rag trực tiếp
src_path = "/content/LegalRAG/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import legal_rag
print("\n✓ Đã kết nối mã nguồn LegalRAG thành công!")

In [ ]:
# 3. Tạo Dense Vector Index (Embedding) trên GPU
import urllib.request
from pathlib import Path
from legal_rag.retrieval.dense import build_dense_index

# Đường dẫn file chunk
chunks_file = Path("/content/LegalRAG/data/chunks/legal_aware_run06/labor_code_2019_legal_aware.jsonl")

# Tự động tải từ GitHub nếu file chưa có sẵn
if not chunks_file.exists():
    chunks_file.parent.mkdir(parents=True, exist_ok=True)
    raw_url = "https://raw.githubusercontent.com/ngminhduc129/LegalRAG/main/data/chunks/legal_aware_run06/labor_code_2019_legal_aware.jsonl"
    print(f"Đang tải file chunks: {raw_url} ...")
    urllib.request.urlretrieve(raw_url, str(chunks_file))

output_dir = Path("/content/LegalRAG/data/indexes/dense")
output_dir.mkdir(parents=True, exist_ok=True)
output_index_path = output_dir / "dense_legal_aware_run06.pkl"

print(f"Đang tính toán Embeddings cho {chunks_file}...")
print(f"Output: {output_index_path}\n")

# Gọi trực tiếp hàm build index (tận dụng GPU PyTorch)
index = build_dense_index(
    chunks_path=chunks_file,
    index_path=output_index_path,
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    batch_size=32,
    max_length=256,
)

print(f"\n✓ Hoàn tất! Đã lưu Dense Index tại: {output_index_path}")
print(f"  Tổng số văn bản: {len(index.documents)}")
print(f"  Kích thước ma trận embeddings: {index.embeddings.shape}")

In [ ]:
# 4. Kiểm tra truy vấn Semantic Search với câu hỏi pháp lý mẫu
test_queries = [
    "Người lao động có quyền đơn phương chấm dứt hợp đồng lao động khi nào?",
    "Thời giờ làm việc bình thường không quá bao nhiêu giờ trong một ngày?",
    "Mức lương tối thiểu được xác lập dựa trên những yếu tố nào?"
]

print("=" * 70)
print("KẾT QUẢ TÌM KIẾM NGỮ NGHĨA (DENSE RETRIEVAL TEST)")
print("=" * 70)

for q_idx, query in enumerate(test_queries, 1):
    print(f"\n[Câu hỏi {q_idx}]: \"{query}\"")
    results = index.search(query, k=3)
    for rank, (doc_id, score, doc) in enumerate(results, 1):
        article_num = doc.get("article", "N/A")
        article_title = doc.get("article_title", "")
        clause_num = doc.get("clause", "")
        point_num = doc.get("point", "")
        
        prefix = f"Điều {article_num}"
        if clause_num:
            prefix += f", Khoản {clause_num}"
        if point_num:
            prefix += f", Điểm {point_num}"
            
        print(f"  Top {rank} (Cosine Sim: {score:.4f}): {prefix} - {article_title}")
        snippet = doc.get("content", "").replace("\n", " ").strip()
        print(f"    Trích đoạn: {snippet[:150]}...")

In [ ]:
# 5. Tải file index về máy (nếu cần)
file_size_mb = output_index_path.stat().st_size / (1024 * 1024)
print(f"✓ File index đã sẵn sàng tại: {output_index_path}")
print(f"  Dung lượng: {file_size_mb:.2f} MB")
print("\nĐể tải file về máy, hãy bỏ dấu comment dòng dưới:")
print("# from google.colab import files; files.download(str(output_index_path))")